<a href="https://colab.research.google.com/github/akshin-kuvshin/numerical-methods/blob/main/lab8/lab8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа №8. Вычисление интегралов методом Гаусса

## Выполнил: Данила Аксёнов, гр. 932420

## Вариант: 1

## **Задание 13**

* Вычислить $ \int_{a}^{b} f(x) \, dx $ с точностью до $ \epsilon = 0.5 * 10^{-5} $ по формуле Гаусса;
* Вывести значение интеграла и число узлов, которое потребовалось для вычисления интеграла с заданной точностью.

In [35]:
import numpy as np
from matplotlib import pyplot as plt
print('Setup completed')

Setup completed


### Исходные данные:

In [36]:
a = 2
b = 3
f = lambda x: np.sqrt(x ** 2 + 2 * x + np.log10(x))
eps = 0.5 * 10 ** -5

### Решение:

Для вычисления интеграла я решил использовать следующую формулу, использующую $ mn $ узлов:

$$ I = \int_{a}^{b} f(x) \, dx = \frac{b - a}{2m} \sum_{i = 1}^{m} \sum_{k = 1}^{n} A_k f \left( a + \frac{b - a}{2m} (t_k + 2i - 1) \right). $$

Здесь отрезок интегрирования $ [a; b] $ делится на $ m $ равных частей, на каждой из которых мы вычисляем интеграл по _формуле Гаусса_ с использованием _многочленом Лежандра_. $ n $ узлов $ t_k $ внутри каждой из $ m $ частей должны быть _корнями многочлена Лежандра_ степени $ n $, для которых **нет общей формулы**.

За $ A_k $ обозначены коэффициенты, вычисляющиеся с использованием многочлена Лежандра степени $ n $ и его корней (про них было сказано выше), как

$$ A_k = \frac{2}{[1 - t_k^2] P'^2_n(t_k)}. $$

Основная проблема, которая ограничивает сверху допустимое для использование значение $ n $ заключается в том, что каждый раз отыскивать все корни $ P_n(t) $ $ - $ многочлена Лежандра степени $ n $ $ - $ задача трудоёмкая как с математической, так и с вычислительной стороны. К счастью, в учебнике (см. таблицу 4.2 на стр. 142-143) есть приближённые значения коэффициентов $ A_k $ и узлов $ t_k $ для всех $ n \le 7 $, поэтому зафиксируем $ n = 7 $, взяв необходимые дополнительные данные из учебника, а заданной точности будет добиваться путём увеличения мощности "внешнего" разбиения $ m $.

Для формулы Гаусса, использующей многочлен Лежандра степени $ n $, исходя из вида её остаточного члена:

$$ R_n(f) = \frac{2^{2n + 1} (n!)^4}{(2n + 1) \left[ (2n)! \right]^3} f^{(2n)}(\xi), \quad \xi \in [-1; 1], \quad - $$

(см. учебник, стр. 141), порядок точности $ p = 2n $, который сохраняется после суммирования по всем $ m $ частям "внешнего" разбиения. Т.к. $ n $ у нас фиксировано, то для остановки будет весьма логично использовать метод Рунге, а именно: мощность "внешнего" разбиения $ m $ прекратим удваивать в тот момент, когда выполнится неравенство:

$$ \frac{\left\lvert I_h - I_{\frac{h}{2}} \right\rvert}{2^p - 1} = \frac{\left\lvert I_h - I_{\frac{h}{2}} \right\rvert}{2^{2n} - 1} \le \epsilon, $$

где $ h $ и $ \frac{h}{2} $ $ - $ шаги "внешнего" разбиения для мощностей $ m $ и $ 2m $ соответственно. В конце, по моему мнению, как и в предыдущей лабораторной работе (предыдущем задании), имеет место использование поправки по методу Рунге (_поправки/экстраполяции Ричардсона_):

$$ I \approx I_{\frac{h}{2}} + \frac{I_h - I_{\frac{h}{2}}}{1 - 2^p} = I_{\frac{h}{2}} + \frac{I_h - I_{\frac{h}{2}}}{1 - 2^{2n}}. $$

К сожалению, "внешние" разбиения мощностей $ m $ и $ 2m $, в общем случае, не имеют общих узлов, в которых для них предстоит вычислять значение функции $ f $, поэтому каждый раз нам придётся пересчитывать значение интеграла "с нуля", т.е. заново.

<u>P.S.</u>: в учебнике корни многочлена Лежандра обозначены как $ x_k $, но я решил поменять их обозначение на $ t_k $, чтобы не путать их с независимой переменной $ x $, т.к., в общем случае, $ x \in [a; b] $, в то время как $ t_k \in [-1; 1] $.

### Дополнительные исходные данные:

In [37]:
n = 7
A = np.array([0.1294849, 0.2797053, 0.3818300, 0.4179591, 0.3818300, 0.2797053, 0.1294849])
t = np.array([-0.9491079, -0.7415311, -0.4058451, 0.0000000, 0.4058451, 0.7415311, 0.9491079])

p = 2 * n

# заранее вычислим выражение (знаменатель), использующееся(-ийся) в методе Рунге
# (как для остановки, так и для поправки), чтобы не пересчитывать его на каждой итерации
d = 2 ** p - 1
d_neg = -d

print('Additional setup completed')

Additional setup completed


In [38]:
# функция, вычисляющая значение интеграла для заданной мощности "внешнего" разбиения m
def calc_I(m):
    c = (b - a) / (2 * m)

    I = 0.0
    for i in range(1, m + 1):
        for k in range(n):
            I += A[k] * f(a + c * (t[k] + 2 * i - 1))
    I *= c
    return I

In [39]:
m = 1
Im = calc_I(m)
while True:
    m2 = m * 2
    Im2 = calc_I(m2)
    if abs(Im - Im2) / d <= eps:
        break
    m = m2
    Im = Im2
print('done.')

done.


In [40]:
d_neg = -d
I = Im2 + (Im - Im2) / d_neg
I, m2, m2 * n

(np.float64(3.4111704500719497), 2, 14)

In [41]:
abs(Im - Im2)

np.float64(3.4979219520892e-10)

_"Расширенная"_ формула Гаусса дала следующий результат: $ \mathbf{\int_a^b f(x)\,dx = \int_2^3 \sqrt{x^2 + 2x + \lg(x)} \, dx \approx 3.41117} $, что согласуется с истиной, где под "истиной" я подразумеваю значение того же интеграла, полученное в предыдущей лабораторной работе (предыдущем задании), которое, в свою очередь, согласуется с результатом, который выдают средства вычисления определённых интегралов, встроенные в SMath Solver (бесплатный аналог Mathcad с открытым исходным кодом, который я использую).

Меня сильно поразил тот факт, что заданная точность $ \epsilon = 0.5 * 10^{-5} $ была достигнута при мощности "внешнего" разбиения $ m = 2 $ (на самом деле достаточно и $ m = 1 $, т.к. соответствующие значения интегралов отличаются друг от друга лишь в 10-ом знаке после запятой), что даёт суммарно $ \mathbf{mn = 2 * 7 = 14} $ **узлов** (опять же, хватило бы и $ 1 * 7 = 7 $ узлов), в которых потребовалось вычислить значение функции $ f $, что говорит о _высокой точности метода Гаусса, использующего многочлен Лежандра_.